In [1]:
import os, gc, glob, warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from scipy.signal import butter, sosfilt

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Speed while preserving determinism: allow TF32 on Ampere+ GPUs (keeps semantics; only tiny FP diffs).
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


# Deterministic DataLoader workers
def seed_worker(worker_id: int):
    worker_seed = (SEED + worker_id) % (2**32)
    np.random.seed(worker_seed)
    torch.manual_seed(worker_seed)


_dl_generator = torch.Generator()
_dl_generator.manual_seed(SEED)




In [2]:
PATH_CANDIDATES = [
    "/kaggle/input/hms-harmful-brain-activity-classification/",
    "/kaggle/input/hms-harmful-brain-activity-classification/hms-harmful-brain-activity-classification/",
]
PATH = None
for p in PATH_CANDIDATES:
    if os.path.exists(os.path.join(p, "train.csv")):
        PATH = p
        break
if PATH is None:
    raise FileNotFoundError(f"Could not find train.csv in any of: {PATH_CANDIDATES}")

TRAIN_CSV = os.path.join(PATH, "train.csv")
TEST_CSV = os.path.join(PATH, "test.csv")
SAMPLE_SUB = os.path.join(PATH, "sample_submission.csv")

EEG_TRAIN_PATH = os.path.join(PATH, "train_eegs/")
EEG_TEST_PATH = os.path.join(PATH, "test_eegs/")

WORK_NORM_CSV = "/kaggle/working/normalized_data.csv"
SUB_PATH = "/kaggle/working/submission.csv"

TARGETS = [
    "seizure_vote",
    "lpd_vote",
    "gpd_vote",
    "lrda_vote",
    "grda_vote",
    "other_vote",
]

print("Using PATH:", PATH)




Using PATH: /kaggle/input/hms-harmful-brain-activity-classification/


In [3]:
df = pd.read_csv(TRAIN_CSV)
print("Train shape:", df.shape)
print("Targets:", TARGETS)
df.head()




Train shape: (96950, 15)
Targets: ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']


,eeg_id,eeg_sub_id,eeg_label_offset_seconds,spectrogram_id,spectrogram_sub_id,spectrogram_label_offset_seconds,label_id,patient_id,expert_consensus,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote
0,1628180742,0,0.0,353733,0,0.0,127492639,42516,Seizure,3,0,0,0,0,0
1,1628180742,1,6.0,353733,1,6.0,3887563113,42516,Seizure,3,0,0,0,0,0
2,1628180742,2,8.0,353733,2,8.0,1142670488,42516,Seizure,3,0,0,0,0,0
3,1628180742,3,18.0,353733,3,18.0,2718991173,42516,Seizure,3,0,0,0,0,0
4,1628180742,4,24.0,353733,4,24.0,3080632009,42516,Seizure,3,0,0,0,0,0


In [4]:
train = df.groupby("eeg_id")[
    ["spectrogram_id", "spectrogram_label_offset_seconds"]
].agg({"spectrogram_id": "first", "spectrogram_label_offset_seconds": "min"})
train.columns = ["spectrogram_id", "min"]

tmp = df.groupby("eeg_id")[["spectrogram_label_offset_seconds"]].agg(
    {"spectrogram_label_offset_seconds": "max"}
)
train["max"] = tmp

tmp = df.groupby("eeg_id")[["patient_id"]].agg("first")
train["patient_id"] = tmp

votes = df[TARGETS].to_numpy(dtype=np.float32, copy=True)
row_sum = votes.sum(axis=1, keepdims=True)
row_sum[row_sum == 0] = 1.0
row_probs = votes / row_sum

row_probs_df = pd.DataFrame(row_probs, columns=TARGETS)
row_probs_df["eeg_id"] = df["eeg_id"].to_numpy()

avg_probs = row_probs_df.groupby("eeg_id")[TARGETS].mean()

y_data = avg_probs.to_numpy(dtype=np.float32, copy=True)
eps = 1e-6
y_data = np.clip(y_data, eps, 1.0)
y_data = y_data / y_data.sum(axis=1, keepdims=True)

for i, t in enumerate(TARGETS):
    train[t] = y_data[:, i]

tmp = df.groupby("eeg_id")[["expert_consensus"]].agg("first")
train["target"] = tmp

train = train.reset_index()
print("Train non-overlapp eeg_id shape:", train.shape)
train.head()




Train non-overlapp eeg_id shape: (15396, 12)


,eeg_id,spectrogram_id,min,max,patient_id,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote,target
0,568657,789577333,0.0,16.0,20654,9.999969e-07,9.999969e-07,2.499992e-01,9.999969e-07,1.666662e-01,5.833315e-01,Other
1,582999,1552638400,0.0,38.0,20230,9.999970e-07,8.571403e-01,9.999970e-07,7.142836e-02,9.999970e-07,7.142836e-02,LPD
2,642382,14960202,1008.0,1032.0,5955,9.999950e-07,9.999950e-07,9.999950e-07,9.999950e-07,9.999950e-07,9.999950e-01,Other
3,1629671,2036345030,0.0,160.0,37481,9.999951e-01,9.999951e-07,9.999951e-07,9.999951e-07,9.999951e-07,9.999951e-07,Seizure
4,1895581,128369999,1138.0,1138.0,47999,7.692284e-02,9.999969e-07,9.999969e-07,9.999969e-07,7.692284e-02,8.461512e-01,Other


In [5]:
eeg_id_col = train["eeg_id"]
prob_cols = train[TARGETS].astype("float32")
label_col = train["target"]

normalized_data = pd.concat(
    [
        eeg_id_col.reset_index(drop=True),
        prob_cols.reset_index(drop=True),
        label_col.reset_index(drop=True),
    ],
    axis=1,
)

normalized_data = normalized_data.drop_duplicates(
    subset=["eeg_id"], keep="first"
).reset_index(drop=True)

normalized_data.to_csv(WORK_NORM_CSV, index=False)
print("Wrote:", WORK_NORM_CSV, "shape:", normalized_data.shape)
normalized_data.head()




Wrote: /kaggle/working/normalized_data.csv shape: (15396, 8)


,eeg_id,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote,target
0,568657,9.999969e-07,9.999969e-07,2.499992e-01,9.999969e-07,1.666662e-01,5.833315e-01,Other
1,582999,9.999970e-07,8.571403e-01,9.999970e-07,7.142836e-02,9.999970e-07,7.142836e-02,LPD
2,642382,9.999950e-07,9.999950e-07,9.999950e-07,9.999950e-07,9.999950e-07,9.999950e-01,Other
3,1629671,9.999951e-01,9.999951e-07,9.999951e-07,9.999951e-07,9.999951e-07,9.999951e-07,Seizure
4,1895581,7.692284e-02,9.999969e-07,9.999969e-07,9.999969e-07,7.692284e-02,8.461512e-01,Other


In [6]:
ycol = TARGETS

# Speed: prefer pyarrow (fast) when available; fall back safely.
try:
    import pyarrow.parquet as pq  # preferred on Kaggle when available
except Exception:
    pq = None


def read_parquet_columns(path, columns):
    # NOTE: Keeping same semantics (reads only requested columns); this is a pure I/O optimization.
    if pq is not None:
        table = pq.read_table(path, columns=columns, use_threads=True)
        return table.to_pandas(self_destruct=True)
    return pd.read_parquet(path, columns=columns)


class EEGDataset(Dataset):
    def __init__(self, csv_file, eeg_path, cache_size=-1):
        self.csv = pd.read_csv(csv_file)
        self.eeg_path = eeg_path
        self.sos = self.butter_bandpass_filter_init()
        self.FEATS = ["Fp1", "T3", "C3", "O1", "Fp2", "C4", "T4", "O2"]

        # NOTE: cache_size=-1 means "unbounded" in this implementation (as originally used).
        self._cache = {}
        self._cache_order = []
        self._cache_size = int(cache_size)

        self.eeg_ids = self.csv["eeg_id"].to_numpy()
        self.labels = self.csv[ycol].to_numpy(dtype=np.float32, copy=True)

        # Speed: timestamps are constant; prebuild once (same values as before).
        self._timestamps = torch.arange(0, 10000, dtype=torch.float32)

    def butter_bandpass_filter_init(self):
        lowcut, highcut, fs, order = 0.5, 40.0, 200.0, 5
        nyq = 0.5 * fs
        low = lowcut / nyq
        high = highcut / nyq
        return butter(order, [low, high], analog=False, btype="band", output="sos")

    def butter_bandpass_filter(self, data):
        return sosfilt(self.sos, data, axis=0)

    def __len__(self):
        return len(self.eeg_ids)

    @staticmethod
    def _mean_impute_inplace(x: np.ndarray) -> np.ndarray:
        if not np.isnan(x).any():
            return x
        col_means = np.nanmean(x, axis=0)
        col_means = np.where(np.isfinite(col_means), col_means, 0.0)
        inds = np.where(np.isnan(x))
        x[inds] = col_means[inds[1]]
        return x

    @staticmethod
    def _zscore_inplace(x: np.ndarray, eps: float = 1e-6) -> np.ndarray:
        mu = x.mean(axis=0, keepdims=True)
        sd = x.std(axis=0, keepdims=True)
        sd = np.where(sd < eps, 1.0, sd)
        x -= mu
        x /= sd
        return x

    def _cache_get(self, key):
        return self._cache.get(key, None)

    def _cache_put(self, key, value):
        if self._cache_size == 0:
            return
        if key in self._cache:
            return
        self._cache[key] = value
        if self._cache_size > 0:
            self._cache_order.append(key)
            if len(self._cache_order) > self._cache_size:
                old = self._cache_order.pop(0)
                self._cache.pop(old, None)

    def __getitem__(self, idx):
        eeg_id = int(self.eeg_ids[idx])

        cached = self._cache_get(eeg_id)
        if cached is not None:
            eeg_data = cached
        else:
            eeg_file_path = os.path.join(self.eeg_path, f"{eeg_id}.parquet")
            try:
                eeg_df = read_parquet_columns(eeg_file_path, columns=self.FEATS)
                eeg_data_np = eeg_df.to_numpy(dtype=np.float32, copy=True)

                eeg_data_np = self._mean_impute_inplace(eeg_data_np)
                eeg_data_np = self.butter_bandpass_filter(eeg_data_np).astype(
                    np.float32, copy=False
                )

                mid_index = eeg_data_np.shape[0] // 2
                start_index = max(0, mid_index - 5000)
                end_index = min(eeg_data_np.shape[0], mid_index + 5000)
                eeg_data_np = eeg_data_np[start_index:end_index]

                target_len = 10000
                cur_len = eeg_data_np.shape[0]
                if cur_len < target_len:
                    pad = np.zeros(
                        (target_len - cur_len, eeg_data_np.shape[1]), dtype=np.float32
                    )
                    eeg_data_np = np.concatenate([eeg_data_np, pad], axis=0)
                elif cur_len > target_len:
                    eeg_data_np = eeg_data_np[:target_len]

                eeg_data_np = self._zscore_inplace(eeg_data_np)
            except Exception:
                eeg_data_np = np.zeros((10000, len(self.FEATS)), dtype=np.float32)

            eeg_data = torch.from_numpy(eeg_data_np.T.copy())
            self._cache_put(eeg_id, eeg_data)

        labels = torch.from_numpy(self.labels[idx].copy())
        return eeg_data, labels, self._timestamps


dataset = EEGDataset(WORK_NORM_CSV, EEG_TRAIN_PATH, cache_size=-1)

# Speed: use multi-worker DataLoader to parallelize parquet I/O + filtering.
# Correctness: deterministic seeds for workers; no shuffle (as original).
_num_workers = min(4, os.cpu_count() or 1)
dataloader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=False,
    num_workers=_num_workers,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=(_num_workers > 0),
    prefetch_factor=2 if _num_workers > 0 else None,
    worker_init_fn=seed_worker if _num_workers > 0 else None,
    generator=_dl_generator,
)

X0, y0, t0 = dataset[0]
print("Sample shapes:", X0.shape, y0.shape, t0.shape, "num_workers:", _num_workers)




Sample shapes: torch.Size([8, 10000]) torch.Size([6]) torch.Size([10000]) num_workers: 4


In [7]:
class TLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers=1, dropout=0.0):
        super(TLSTM, self).__init__()
        self.lstm = nn.LSTM(
            input_size,
            hidden_size,
            num_layers,
            dropout=dropout,
            bidirectional=True,
            batch_first=True,
        )

    def forward(self, x, timestamps):
        output, (h_n, c_n) = self.lstm(x)
        return output, (h_n, c_n)


class EEGNet(nn.Module):
    def __init__(self, in_channels=8, num_classes=6):
        super(EEGNet, self).__init__()
        self.conv1 = nn.Conv1d(in_channels, 32, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm1d(32)
        self.relu1 = nn.ReLU(inplace=True)
        self.pool1 = nn.MaxPool1d(kernel_size=2, stride=2)

        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm1d(64)
        self.relu2 = nn.ReLU(inplace=True)
        self.pool2 = nn.MaxPool1d(kernel_size=2, stride=2)

        self.tlstm = TLSTM(input_size=64, hidden_size=128)

        self.attention = nn.Sequential(
            nn.Linear(128 * 2, 64),
            nn.Tanh(),
            nn.Linear(64, 1),
        )

        self.fc = nn.Linear(128 * 2, num_classes)

    def forward(self, x, timestamps):
        x = self.pool1(self.relu1(self.bn1(self.conv1(x))))
        x = self.pool2(self.relu2(self.bn2(self.conv2(x))))

        x = x.permute(0, 2, 1)

        x, _ = self.tlstm(x, timestamps)

        att_weights = F.softmax(self.attention(x), dim=1)  # [B, T', 1]
        x = torch.sum(att_weights * x, dim=1)  # [B, 256]
        x = self.fc(x)
        return x




In [8]:
input_channels = 8
num_classes = 6

model = EEGNet(in_channels=input_channels, num_classes=num_classes)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.KLDivLoss(reduction="batchmean")
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 5
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels, timestamps in dataloader:
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        timestamps = timestamps.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(inputs, timestamps=timestamps)
        log_probs = F.log_softmax(logits, dim=1)
        loss = criterion(log_probs, labels)
        loss.backward()
        optimizer.step()
        running_loss += float(loss.item())

    print(f"Epoch {epoch+1}, Loss: {running_loss / len(dataloader):.6f}")




Epoch 1, Loss: 1.035557
Epoch 2, Loss: 0.834835
Epoch 3, Loss: 0.777523
Epoch 4, Loss: 0.736534
Epoch 5, Loss: 0.706681


In [9]:
class TestEEGDataset(Dataset):
    def __init__(self, eeg_ids, eeg_path, cache_size=-1):
        self.eeg_ids = np.asarray(eeg_ids)
        self.eeg_path = eeg_path
        self.sos = self.butter_bandpass_filter_init()
        self.FEATS = ["Fp1", "T3", "C3", "O1", "Fp2", "C4", "T4", "O2"]

        self._cache = {}
        self._cache_order = []
        self._cache_size = int(cache_size)

        # Speed: constant timestamps; prebuild once.
        self._timestamps = torch.arange(0, 10000, dtype=torch.float32)

    def __len__(self):
        return len(self.eeg_ids)

    def butter_bandpass_filter_init(self):
        lowcut, highcut, fs, order = 0.5, 40.0, 200.0, 5
        nyq = 0.5 * fs
        low = lowcut / nyq
        high = highcut / nyq
        return butter(order, [low, high], analog=False, btype="band", output="sos")

    def butter_bandpass_filter(self, data):
        return sosfilt(self.sos, data, axis=0)

    @staticmethod
    def _mean_impute_inplace(x: np.ndarray) -> np.ndarray:
        if not np.isnan(x).any():
            return x
        col_means = np.nanmean(x, axis=0)
        col_means = np.where(np.isfinite(col_means), col_means, 0.0)
        inds = np.where(np.isnan(x))
        x[inds] = col_means[inds[1]]
        return x

    @staticmethod
    def _zscore_inplace(x: np.ndarray, eps: float = 1e-6) -> np.ndarray:
        mu = x.mean(axis=0, keepdims=True)
        sd = x.std(axis=0, keepdims=True)
        sd = np.where(sd < eps, 1.0, sd)
        x -= mu
        x /= sd
        return x

    def _cache_get(self, key):
        return self._cache.get(key, None)

    def _cache_put(self, key, value):
        if self._cache_size == 0:
            return
        if key in self._cache:
            return
        self._cache[key] = value
        if self._cache_size > 0:
            self._cache_order.append(key)
            if len(self._cache_order) > self._cache_size:
                old = self._cache_order.pop(0)
                self._cache.pop(old, None)

    def __getitem__(self, idx):
        eeg_id = int(self.eeg_ids[idx])

        cached = self._cache_get(eeg_id)
        if cached is not None:
            return cached

        eeg_file_path = os.path.join(self.eeg_path, f"{eeg_id}.parquet")
        try:
            eeg_df = read_parquet_columns(eeg_file_path, columns=self.FEATS)
            eeg_data_np = eeg_df.to_numpy(dtype=np.float32, copy=True)

            eeg_data_np = self._mean_impute_inplace(eeg_data_np)
            eeg_data_np = self.butter_bandpass_filter(eeg_data_np).astype(
                np.float32, copy=False
            )

            mid_index = eeg_data_np.shape[0] // 2
            start_index = max(0, mid_index - 5000)
            end_index = min(eeg_data_np.shape[0], mid_index + 5000)
            eeg_data_np = eeg_data_np[start_index:end_index]

            target_len = 10000
            cur_len = eeg_data_np.shape[0]
            if cur_len < target_len:
                pad = np.zeros(
                    (target_len - cur_len, eeg_data_np.shape[1]), dtype=np.float32
                )
                eeg_data_np = np.concatenate([eeg_data_np, pad], axis=0)
            elif cur_len > target_len:
                eeg_data_np = eeg_data_np[:target_len]

            eeg_data_np = self._zscore_inplace(eeg_data_np)
        except Exception:
            eeg_data_np = np.zeros((10000, len(self.FEATS)), dtype=np.float32)

        eeg_data = torch.from_numpy(eeg_data_np.T.copy())
        self._cache_put(eeg_id, eeg_data)
        return eeg_data


sample_sub = pd.read_csv(SAMPLE_SUB)

# Correctness + fixes "length mismatch":
# Kaggle scoring expects predictions for eeg_ids in test.csv. We generate for that set,
# then reindex to sample_submission order to guarantee identical length/order.
test_df = pd.read_csv(TEST_CSV)
test_eeg_ids = test_df["eeg_id"].to_numpy()

testdataset = TestEEGDataset(test_eeg_ids, EEG_TEST_PATH, cache_size=-1)

_num_workers_test = min(4, os.cpu_count() or 1)
test_dataloader = DataLoader(
    testdataset,
    batch_size=32,
    shuffle=False,
    num_workers=_num_workers_test,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=(_num_workers_test > 0),
    prefetch_factor=2 if _num_workers_test > 0 else None,
    worker_init_fn=seed_worker if _num_workers_test > 0 else None,
    generator=_dl_generator,
)

model.eval()
predictions = []
with torch.no_grad():
    # Speed: reuse a single base timestamps tensor and expand (same values as original).
    base_timestamps = torch.arange(
        0, 10000, dtype=torch.float32, device=device
    ).unsqueeze(0)
    for inputs in test_dataloader:
        inputs = inputs.to(device, non_blocking=True)
        bsz = inputs.size(0)
        timestamps = base_timestamps.expand(bsz, -1)
        logits = model(inputs, timestamps=timestamps)
        probs = torch.softmax(logits, dim=1)
        predictions.append(probs.cpu().numpy())

predictions = np.concatenate(predictions, axis=0)
print(
    "Predictions shape:",
    predictions.shape,
    "Expected test:",
    len(test_eeg_ids),
    "Sample_sub:",
    len(sample_sub),
)




Predictions shape: (9850, 6) Expected test: 9850 Sample_sub: 9850


In [10]:
if "normalized_data" not in globals():
    normalized_data = pd.read_csv(WORK_NORM_CSV)

pred = predictions.astype("float64")
eps = 1e-6
pred = np.clip(pred, eps, 1.0)
pred = pred / pred.sum(axis=1, keepdims=True)

prior = normalized_data[TARGETS].mean(axis=0).to_numpy(dtype=np.float64)
prior = np.clip(prior, eps, 1.0)
prior = prior / prior.sum()

alpha = 0.42  # keep as-is (calibration already chosen to be closer to target than raw)
pred = (1.0 - alpha) * pred + alpha * prior[None, :]
pred = np.clip(pred, eps, 1.0)
pred = pred / pred.sum(axis=1, keepdims=True)

beta = 0.09  # keep as-is
uniform = np.full((1, len(TARGETS)), 1.0 / len(TARGETS), dtype=np.float64)
pred = (1.0 - beta) * pred + beta * uniform
pred = np.clip(pred, eps, 1.0)
pred = pred / pred.sum(axis=1, keepdims=True)

# Build predictions dataframe indexed by test eeg_id, then reindex to sample_submission eeg_id order.
pred_df = pd.DataFrame(pred, columns=TARGETS)
pred_df["eeg_id"] = test_eeg_ids

sub = sample_sub[["eeg_id"]].copy()
sub = sub.merge(pred_df, on="eeg_id", how="left")

# If any eeg_id missing (shouldn't happen), fill with prior (deterministic and valid).
missing = sub[TARGETS].isna().any(axis=1)
if bool(missing.any()):
    sub.loc[missing, TARGETS] = prior[None, :]

row_sums = sub[TARGETS].sum(axis=1).to_numpy()
row_sums[row_sums == 0] = 1.0
sub[TARGETS] = sub[TARGETS].div(row_sums, axis=0)

sub.to_csv(SUB_PATH, index=False)
print("Wrote submission:", SUB_PATH)
print("Submission shape:", sub.shape)
print(
    "Min/Max row sum:", sub[TARGETS].sum(axis=1).min(), sub[TARGETS].sum(axis=1).max()
)
sub.head()




Wrote submission: /kaggle/working/submission.csv
Submission shape: (778942, 7)
Min/Max row sum: 0.9999999999999998 1.0000000000000002


,eeg_id,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote
0,2578018731,0.093002,0.074226,0.131009,0.047903,0.33945,0.31441
1,2578018731,0.093002,0.074226,0.131009,0.047903,0.33945,0.31441
2,2578018731,0.093002,0.074226,0.131009,0.047903,0.33945,0.31441
3,2578018731,0.093002,0.074226,0.131009,0.047903,0.33945,0.31441
4,2578018731,0.093002,0.074226,0.131009,0.047903,0.33945,0.31441


In [11]:
assert list(sub.columns) == ["eeg_id"] + TARGETS
assert np.isfinite(sub[TARGETS].values).all()
assert len(sub) == len(sample_sub)
assert np.allclose(sub[TARGETS].sum(axis=1).values, 1.0, atol=1e-6)
assert np.array_equal(sub["eeg_id"].to_numpy(), sample_sub["eeg_id"].to_numpy())
print("All checks passed.")

AssertionError: 